# Anotacoes

Perdas com valores positivos
- Encontrado OPs com valores positivos para perdas
- Necessário aplicar tratamento transformando perdas com valores positivos para negativos
- Recalculas pontos de medição apos tratamento de perdas positivas
- Status: APLICADO
Atenção:
- Há OPs que firacam com valores negativos/diferentes com base "desempenho_empacotamento_diario.xlsx"
- Motivos:
    - "Rejeito total AF" talvez seja ponto de medição

# Importando bibliotecas

In [18]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import ipywidgets as widgets
from IPython.display import display, Markdown

from plotly.subplots import make_subplots

# Importando dados

In [19]:
df = pd.read_excel('../data/raw/Zonas e Perdas (sensor).xlsx')
df_desemp_emp_diario = pd.read_excel('../data/processed/desempenho_empacotamento_diario.xlsx')

c:\Users\evosystem03.ti\Documents\Demanda Carteira\Projeto\predicao-carteira-wheaton\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


# Visualizando dados

In [20]:
df

,Data Wheaton (dia),Forno,Maquina,Prefixo,Ordem Producao,Unnamed: 5,Unnamed: 6
0,2026-08-10,1,10,MB -1398-S,198594,Qtd Gotas Cortadas,23039.000000
1,2026-08-10,1,10,MB -1398-S,198594,Paradas de seção,-38.000000
2,2026-08-10,1,10,MB -1398-S,198594,Rejeito seções,-532.000000
3,2026-08-10,1,10,MB -1398-S,198594,Perdas Maq-Rec,-666.000000
4,2026-08-10,1,10,MB -1398-S,198594,Qtd Entrada Recoz.,21803.000000
...,...,...,...,...,...,...,...
415,2026-08-10,B,B7,MB -1585-S,198546,%Perdas Recozi.,-0.040799
416,2026-08-10,B,B7,MB -1585-S,198546,%Entrada AF,0.872265
417,2026-08-10,B,B7,MB -1585-S,198546,%Rejeição EA,-0.037930
418,2026-08-10,B,B7,MB -1585-S,198546,%Rejeito AF,-0.138531


# Tratando dados

In [21]:
# visualizando variaveis e seus tipos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Data Wheaton (dia)  420 non-null    datetime64[us]
 1   Forno               420 non-null    str           
 2   Maquina             420 non-null    str           
 3   Prefixo             420 non-null    str           
 4   Ordem Producao      420 non-null    int64         
 5   Unnamed: 5          420 non-null    str           
 6   Unnamed: 6          420 non-null    float64       
dtypes: datetime64[us](1), float64(1), int64(1), str(4)
memory usage: 35.3 KB


In [22]:
# renomeando variaveis ja conhecidas com nome Unnamed
df = df.rename(columns={'Unnamed: 5': 'Zonas', 'Unnamed: 6': 'Qtd Frascos'})
df.columns

Index(['Data Wheaton (dia)', 'Forno', 'Maquina', 'Prefixo', 'Ordem Producao',
       'Zonas', 'Qtd Frascos'],
      dtype='str')

In [23]:
'''# filtrando df para ter apenas linhas cuja Zona e um percentual
df = df[df['Zonas'].str.contains('%')]'''

"# filtrando df para ter apenas linhas cuja Zona e um percentual\ndf = df[df['Zonas'].str.contains('%')]"

In [24]:
# tratando as perdas registradas com valor positivo e recalculando os pontos de medicao

# zonas que representam perda de produto: por regra de negocio devem ser sempre negativas
zonas_perdas = [
    'Paradas de seção',
    'Rejeito seções',
    'Perdas Maq-Rec',
    'Perdas Recozimento',
    'Perdas EA',
    'Rejeito total AF',
]

# pontos de medicao: nao sao perdas, e sim o que restou apos as perdas acumuladas ate ali
zonas_medicao = ['Qtd Entrada Recoz.', 'Qtd Entrada AF', 'Qtd Empacotado']

# 1) toda perda registrada como positiva passa a ser negativa
perdas_positivas = df['Zonas'].isin(zonas_perdas) & (df['Qtd Frascos'] > 0)
df.loc[perdas_positivas, 'Qtd Frascos'] = -df.loc[perdas_positivas, 'Qtd Frascos']

# 2) visao larga (uma linha por OP) para aplicar a cascata
qtd_op = df[df['Zonas'].isin(['Qtd Gotas Cortadas'] + zonas_perdas + zonas_medicao)].pivot_table(
    index='Ordem Producao', columns='Zonas', values='Qtd Frascos', aggfunc='sum'
)

# 3) recalculando os pontos de medicao na ordem do processo
#    'Perdas EA' (rejeicao da maquina automatizada) e 'Rejeito total AF' (rejeicao manual)
#    sao perdas distintas do acabamento final, entao as duas somam
qtd_op['Qtd Entrada Recoz.'] = (
    qtd_op['Qtd Gotas Cortadas']
    + qtd_op['Paradas de seção']
    + qtd_op['Rejeito seções']
    + qtd_op['Perdas Maq-Rec']
)
qtd_op['Qtd Entrada AF'] = qtd_op['Qtd Entrada Recoz.'] + qtd_op['Perdas Recozimento']
qtd_op['Qtd Empacotado'] = qtd_op['Qtd Entrada AF'] + qtd_op['Perdas EA'] + qtd_op['Rejeito total AF']

# 4) devolvendo os valores recalculados para o df em formato longo
for zona in zonas_medicao:
    linhas_zona = df['Zonas'] == zona
    df.loc[linhas_zona, 'Qtd Frascos'] = df.loc[linhas_zona, 'Ordem Producao'].map(qtd_op[zona])

qtd_ops_corrigidas = df.loc[perdas_positivas, 'Ordem Producao'].nunique()
display(Markdown(f'### {int(perdas_positivas.sum())} perda(s) positiva(s) corrigida(s) em {qtd_ops_corrigidas} OP(s)'))

# OPs que seguem impossiveis mesmo apos o tratamento: ponto de medicao negativo
medicao_negativa = qtd_op[(qtd_op[zonas_medicao] < 0).any(axis=1)]

display(Markdown(f'### {len(medicao_negativa)} OP(s) com ponto de medicao negativo apos o tratamento (revisar na origem)'))
display(medicao_negativa[zonas_medicao])
print()

### 8 perda(s) positiva(s) corrigida(s) em 5 OP(s)

### 2 OP(s) com ponto de medicao negativo apos o tratamento (revisar na origem)

Zonas,Qtd Entrada Recoz.,Qtd Entrada AF,Qtd Empacotado
Ordem Producao,,,
198663,5506.0,5253.0,-2129.0
198693,24873.0,13360.0,-21032.0


In [25]:
display(df)

,Data Wheaton (dia),Forno,Maquina,Prefixo,Ordem Producao,Zonas,Qtd Frascos
0,2026-08-10,1,10,MB -1398-S,198594,Qtd Gotas Cortadas,23039.000000
1,2026-08-10,1,10,MB -1398-S,198594,Paradas de seção,-38.000000
2,2026-08-10,1,10,MB -1398-S,198594,Rejeito seções,-532.000000
3,2026-08-10,1,10,MB -1398-S,198594,Perdas Maq-Rec,-666.000000
4,2026-08-10,1,10,MB -1398-S,198594,Qtd Entrada Recoz.,21803.000000
...,...,...,...,...,...,...,...
415,2026-08-10,B,B7,MB -1585-S,198546,%Perdas Recozi.,-0.040799
416,2026-08-10,B,B7,MB -1585-S,198546,%Entrada AF,0.872265
417,2026-08-10,B,B7,MB -1585-S,198546,%Rejeição EA,-0.037930
418,2026-08-10,B,B7,MB -1585-S,198546,%Rejeito AF,-0.138531


# Analises

## Visualizando perdas por OP

In [26]:
# um display do df filtrado para cada OP, na ordem em que aparecem na base
for op in df['Ordem Producao'].unique():
    df_op = df[df['Ordem Producao'] == op]

    display(Markdown(f'### OP {op}'))
    display(df_op)
    print()

### OP 198594

,Data Wheaton (dia),Forno,Maquina,Prefixo,Ordem Producao,Zonas,Qtd Frascos
0,2026-08-10,1,10,MB -1398-S,198594,Qtd Gotas Cortadas,23039.000000
1,2026-08-10,1,10,MB -1398-S,198594,Paradas de seção,-38.000000
2,2026-08-10,1,10,MB -1398-S,198594,Rejeito seções,-532.000000
3,2026-08-10,1,10,MB -1398-S,198594,Perdas Maq-Rec,-666.000000
4,2026-08-10,1,10,MB -1398-S,198594,Qtd Entrada Recoz.,21803.000000
5,2026-08-10,1,10,MB -1398-S,198594,Perdas Recozimento,-1651.000000
6,2026-08-10,1,10,MB -1398-S,198594,Qtd Entrada AF,20152.000000
7,2026-08-10,1,10,MB -1398-S,198594,Perdas EA,0.000000
8,2026-08-10,1,10,MB -1398-S,198594,Rejeito total AF,-6568.000000
9,2026-08-10,1,10,MB -1398-S,198594,Qtd Empacotado,13584.000000


### OP 198660

,Data Wheaton (dia),Forno,Maquina,Prefixo,Ordem Producao,Zonas,Qtd Frascos
20,2026-08-10,1,11,LB -0586-S,198660,Qtd Gotas Cortadas,37567.000000
21,2026-08-10,1,11,LB -0586-S,198660,Paradas de seção,-1014.000000
22,2026-08-10,1,11,LB -0586-S,198660,Rejeito seções,-2086.000000
23,2026-08-10,1,11,LB -0586-S,198660,Perdas Maq-Rec,-526.000000
24,2026-08-10,1,11,LB -0586-S,198660,Qtd Entrada Recoz.,33941.000000
25,2026-08-10,1,11,LB -0586-S,198660,Perdas Recozimento,-3599.000000
26,2026-08-10,1,11,LB -0586-S,198660,Qtd Entrada AF,30342.000000
27,2026-08-10,1,11,LB -0586-S,198660,Perdas EA,-951.000000
28,2026-08-10,1,11,LB -0586-S,198660,Rejeito total AF,-6270.000000
29,2026-08-10,1,11,LB -0586-S,198660,Qtd Empacotado,23121.000000


### OP 198592

...

,Data Wheaton (dia),Forno,Maquina,Prefixo,Ordem Producao,Zonas,Qtd Frascos
360,2026-08-10,B,B5,SB -1471-SN,198613,Qtd Gotas Cortadas,335751.000000
361,2026-08-10,B,B5,SB -1471-SN,198613,Paradas de seção,-77613.000000
362,2026-08-10,B,B5,SB -1471-SN,198613,Rejeito seções,-5089.000000
363,2026-08-10,B,B5,SB -1471-SN,198613,Perdas Maq-Rec,-7364.000000
364,2026-08-10,B,B5,SB -1471-SN,198613,Qtd Entrada Recoz.,245685.000000
365,2026-08-10,B,B5,SB -1471-SN,198613,Perdas Recozimento,-3994.000000
366,2026-08-10,B,B5,SB -1471-SN,198613,Qtd Entrada AF,241691.000000
367,2026-08-10,B,B5,SB -1471-SN,198613,Perdas EA,-49021.000000
368,2026-08-10,B,B5,SB -1471-SN,198613,Rejeito total AF,-56739.000000
369,2026-08-10,B,B5,SB -1471-SN,198613,Qtd Empacotado,135931.000000


### OP 198693

,Data Wheaton (dia),Forno,Maquina,Prefixo,Ordem Producao,Zonas,Qtd Frascos
380,2026-08-10,B,B6,PL -5814-,198693,Qtd Gotas Cortadas,43187.000000
381,2026-08-10,B,B6,PL -5814-,198693,Paradas de seção,-17592.000000
382,2026-08-10,B,B6,PL -5814-,198693,Rejeito seções,0.000000
383,2026-08-10,B,B6,PL -5814-,198693,Perdas Maq-Rec,-722.000000
384,2026-08-10,B,B6,PL -5814-,198693,Qtd Entrada Recoz.,24873.000000
385,2026-08-10,B,B6,PL -5814-,198693,Perdas Recozimento,-11513.000000
386,2026-08-10,B,B6,PL -5814-,198693,Qtd Entrada AF,13360.000000
387,2026-08-10,B,B6,PL -5814-,198693,Perdas EA,0.000000
388,2026-08-10,B,B6,PL -5814-,198693,Rejeito total AF,-34392.000000
389,2026-08-10,B,B6,PL -5814-,198693,Qtd Empacotado,-21032.000000


### OP 198546

,Data Wheaton (dia),Forno,Maquina,Prefixo,Ordem Producao,Zonas,Qtd Frascos
400,2026-08-10,B,B7,MB -1585-S,198546,Qtd Gotas Cortadas,118190.000000
401,2026-08-10,B,B7,MB -1585-S,198546,Paradas de seção,-1244.000000
402,2026-08-10,B,B7,MB -1585-S,198546,Rejeito seções,-3587.000000
403,2026-08-10,B,B7,MB -1585-S,198546,Perdas Maq-Rec,-5444.000000
404,2026-08-10,B,B7,MB -1585-S,198546,Qtd Entrada Recoz.,107915.000000
405,2026-08-10,B,B7,MB -1585-S,198546,Perdas Recozimento,-4822.000000
406,2026-08-10,B,B7,MB -1585-S,198546,Qtd Entrada AF,103093.000000
407,2026-08-10,B,B7,MB -1585-S,198546,Perdas EA,-4483.000000
408,2026-08-10,B,B7,MB -1585-S,198546,Rejeito total AF,-16373.000000
409,2026-08-10,B,B7,MB -1585-S,198546,Qtd Empacotado,82237.000000


## Validando se todas as zonas de perdas possuem valores negativos

In [27]:
# zonas que representam perda de produto: por regra de negocio devem ser sempre negativas
zonas_perdas = [
    'Paradas de seção',
    'Rejeito seções',
    'Perdas Maq-Rec',
    'Perdas Recozimento',
    'Perdas EA',
    'Rejeito total AF',
]

# linhas de perda que estao com Qtd Frascos positivo (inconsistentes)
perdas_positivas = df[df['Zonas'].isin(zonas_perdas) & (df['Qtd Frascos'] > 0)]

# uma linha por OP, com o conjunto das zonas problematicas
inconsistencias = (
    perdas_positivas
    .groupby('Ordem Producao')['Zonas']
    .apply(set)
    .reset_index()
    .rename(columns={'Zonas': 'Zona de Perda c/ Qtd Frascos positivo'})
)

display(Markdown(f'### {len(inconsistencias)} OP(s) inconsistente(s) de {df["Ordem Producao"].nunique()} no total'))
display(inconsistencias)
print()

# visualizando o df completo de cada OP inconsistente
ops_inconsistentes = inconsistencias['Ordem Producao']
df_inconsistentes = df[df['Ordem Producao'].isin(ops_inconsistentes)]

for op in ops_inconsistentes:
    zonas = inconsistencias.loc[inconsistencias['Ordem Producao'] == op,
                                'Zona de Perda c/ Qtd Frascos positivo'].iloc[0]

    display(Markdown(f'### OP {op} — zonas positivas: {", ".join(sorted(zonas))}'))
    display(df_inconsistentes[df_inconsistentes['Ordem Producao'] == op])
    print()

### 0 OP(s) inconsistente(s) de 21 no total

,Ordem Producao,Zona de Perda c/ Qtd Frascos positivo


## Graficos

In [28]:
# grafico 1: cascata (waterfall), um plot para cada OP do dataframe
# obs: o plotly.express nao tem waterfall, esse tipo so existe no graph_objects (go)

# cada etapa com o seu papel na cascata:
#   absolute = ponto de partida | relative = perda | total = ponto de medicao (subtotal)
# no acabamento final ha duas rejeicoes: 'Perdas EA' pela maquina automatizada
# e 'Rejeito total AF' manual, feita por pessoas
etapas_cascata = [
    ('Qtd Gotas Cortadas', 'absolute'),
    ('Paradas de seção', 'relative'),
    ('Rejeito seções', 'relative'),
    ('Perdas Maq-Rec', 'relative'),
    ('Qtd Entrada Recoz.', 'total'),
    ('Perdas Recozimento', 'relative'),
    ('Qtd Entrada AF', 'total'),
    ('Perdas EA', 'relative'),
    ('Rejeito total AF', 'relative'),
    ('Qtd Empacotado', 'total'),
]

etapas = [etapa for etapa, _ in etapas_cascata]
medidas = [medida for _, medida in etapas_cascata]

for op in qtd_op.index:
    valores = [qtd_op.loc[op, etapa] for etapa in etapas]
    rendimento = qtd_op.loc[op, 'Qtd Empacotado'] / qtd_op.loc[op, 'Qtd Gotas Cortadas'] * 100

    # OPs com ponto de medicao negativo seguem com dado corrompido na origem
    alerta = '' if (qtd_op.loc[op, zonas_medicao] >= 0).all() else '  **(dado inconsistente na origem)**'

    display(Markdown(f'### OP {op} — rendimento {rendimento:.1f}%{alerta}'))

    fig = go.Figure(go.Waterfall(
        orientation='v',
        measure=medidas,
        x=etapas,
        y=valores,
        text=[f'{valor:,.0f}'.replace(',', '.') for valor in valores],
        textposition='outside',
        connector={'line': {'color': '#8a8a85', 'dash': 'dot', 'width': 1}},
        decreasing={'marker': {'color': '#e34948'}},
        increasing={'marker': {'color': '#2a78d6'}},
        totals={'marker': {'color': '#2a78d6'}},
    ))

    fig.update_layout(
        title=f'Cascata de perdas ao longo do processo — OP {op}',
        xaxis_title='Etapa do processo',
        yaxis_title='Qtd Frascos',
        template='plotly_white',
        showlegend=False,
        height=460,
    )

    fig.show()
    print()

# o grafico seguinte usa apenas as OPs com os pontos de medicao consistentes
qtd_op_valida = qtd_op[(qtd_op[zonas_medicao] >= 0).all(axis=1)]

### OP 198135 — rendimento 79.9%

### OP 198176 — rendimento 62.0%

### OP 198456 — rendimento 57.7%

### OP 198546 — rendimento 69.6%

### OP 198567 — rendimento 66.7%

...

### OP 198694 — rendimento 57.3%

### OP 198698 — rendimento 16.7%

### OP 198701 — rendimento 17.4%

### OP 198704 — rendimento 35.7%

### OP 198707 — rendimento 74.6%

In [29]:
# grafico 2 (recomendado): composicao de cada OP, ordenada pelo rendimento
# cada barra soma 100% das Gotas Cortadas, repartidas entre o que foi empacotado e cada perda
# responde "qual OP perdeu mais e por qual zona", que a cascata sozinha nao mostra

zonas_composicao = [
    'Qtd Empacotado',
    'Paradas de seção',
    'Rejeito seções',
    'Perdas Maq-Rec',
    'Perdas Recozimento',
    'Perdas EA',
    'Rejeito total AF',
]

composicao = (
    qtd_op_valida[zonas_composicao]
    .abs()
    .div(qtd_op_valida['Qtd Gotas Cortadas'], axis=0)
    .mul(100)
)

# pior rendimento no topo do grafico
ordem_ops = composicao['Qtd Empacotado'].sort_values().index.astype(str).tolist()

composicao_longa = composicao.reset_index().melt(
    id_vars='Ordem Producao',
    var_name='Zonas',
    value_name='% sobre Gotas Cortadas',
)
composicao_longa['Ordem Producao'] = composicao_longa['Ordem Producao'].astype(str)

fig = px.bar(
    composicao_longa,
    x='% sobre Gotas Cortadas',
    y='Ordem Producao',
    color='Zonas',
    orientation='h',
    category_orders={'Ordem Producao': ordem_ops, 'Zonas': zonas_composicao},
    color_discrete_sequence=['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7'],
    title='Composicao da producao por OP (% sobre Qtd Gotas Cortadas)',
    template='plotly_white',
    height=760,
)

# separacao de 2px entre os segmentos empilhados
fig.update_traces(marker_line_color='#fcfcfb', marker_line_width=2)
fig.update_layout(legend_title_text='', bargap=0.25)

fig.show()

# visao em tabela do mesmo dado (o grafico tem cores de baixo contraste, a tabela garante a leitura)
display(Markdown('### Composicao por OP em % (mesma ordem do grafico)'))
display(composicao.loc[[int(op) for op in ordem_ops]].round(1))
print()

### Composicao por OP em % (mesma ordem do grafico)

Zonas,Qtd Empacotado,Paradas de seção,Rejeito seções,Perdas Maq-Rec,Perdas Recozimento,Perdas EA,Rejeito total AF
Ordem Producao,,,,,,,
198698,16.7,8.7,6.3,26.6,8.3,16.7,16.7
198701,17.4,0.4,0.0,14.9,27.7,10.0,29.6
198704,35.7,1.6,0.0,9.2,23.4,7.5,22.5
198613,40.5,23.1,1.5,2.2,1.2,14.6,16.9
198694,57.3,1.4,4.4,2.7,8.4,10.7,15.1
198456,57.7,18.6,4.9,2.7,3.6,4.7,7.8
198592,58.2,0.7,4.3,5.9,3.0,10.6,17.4
198594,59.0,0.2,2.3,2.9,7.2,0.0,28.5
198660,61.5,2.7,5.6,1.4,9.6,2.5,16.7


## Validando o % empacotado entre as bases Zonas e Perdas e Desempenho de empacotamento

In [30]:
# comparando o % empacotado por OP entre as duas bases

# a base de desempenho traz o percentual em fracao (0.87 = 87%), por isso o *100
desempenho_pct = df_desemp_emp_diario.set_index('OP Vertech')['Empacotado %'] * 100

# na base de zonas o percentual e calculado: (Qtd Empacotado / Qtd Gotas Cortadas) * 100
zonas_pct = qtd_op['Qtd Empacotado'] / qtd_op['Qtd Gotas Cortadas'] * 100

# OPs que nao existem nas duas bases nao tem como ser comparadas
so_em_zonas = qtd_op.index.difference(desempenho_pct.index)
so_em_desempenho = desempenho_pct.index.difference(qtd_op.index)

comparacao = pd.DataFrame({
    '% Empacotado (Zonas e Perdas)': zonas_pct,
    '% Empacotado (Desempenho)': desempenho_pct,
})
comparacao.index.name = 'Ordem Producao'
comparacao['Diferença (pp)'] = (
    comparacao['% Empacotado (Zonas e Perdas)'] - comparacao['% Empacotado (Desempenho)']
)

# tolerancia em pontos percentuais, para absorver arredondamento entre os sistemas
tolerancia_pp = 0.1

comparacao['Situação'] = 'DIVERGENTE'
comparacao.loc[comparacao['Diferença (pp)'].abs() <= tolerancia_pp, 'Situação'] = 'IGUAL'
comparacao.loc[comparacao['Diferença (pp)'].isna(), 'Situação'] = 'SEM PAR'

# maiores divergencias primeiro
comparacao = comparacao.sort_values('Diferença (pp)', key=abs, ascending=False).round(2)

qtd_iguais = int((comparacao['Situação'] == 'IGUAL').sum())
qtd_divergentes = int((comparacao['Situação'] == 'DIVERGENTE').sum())

display(Markdown(
    f'### {qtd_iguais} igual(is) e {qtd_divergentes} divergente(s) de {len(comparacao)} OPs '
    f'(tolerancia de {tolerancia_pp} pp)'
))
display(comparacao)
print()

if len(so_em_zonas) or len(so_em_desempenho):
    display(Markdown('### OPs sem par entre as bases'))
    display(pd.DataFrame({
        'Ordem Producao': list(so_em_zonas) + list(so_em_desempenho),
        'Presente apenas em': ['Zonas e Perdas'] * len(so_em_zonas) + ['Desempenho'] * len(so_em_desempenho),
    }))
    print()

### 2 igual(is) e 19 divergente(s) de 21 OPs (tolerancia de 0.1 pp)

,% Empacotado (Zonas e Perdas),% Empacotado (Desempenho),Diferença (pp),Situação
Ordem Producao,,,,
198693,-48.70,87.13,-135.83,DIVERGENTE
198663,-30.20,94.67,-124.87,DIVERGENTE
198701,17.40,38.94,-21.54,DIVERGENTE
198613,40.49,61.05,-20.57,DIVERGENTE
198456,57.70,77.39,-19.69,DIVERGENTE
198698,16.74,33.27,-16.53,DIVERGENTE
198567,66.68,79.63,-12.95,DIVERGENTE
198704,35.74,46.93,-11.18,DIVERGENTE
198592,58.16,68.71,-10.55,DIVERGENTE


In [31]:
df

,Data Wheaton (dia),Forno,Maquina,Prefixo,Ordem Producao,Zonas,Qtd Frascos
0,2026-08-10,1,10,MB -1398-S,198594,Qtd Gotas Cortadas,23039.000000
1,2026-08-10,1,10,MB -1398-S,198594,Paradas de seção,-38.000000
2,2026-08-10,1,10,MB -1398-S,198594,Rejeito seções,-532.000000
3,2026-08-10,1,10,MB -1398-S,198594,Perdas Maq-Rec,-666.000000
4,2026-08-10,1,10,MB -1398-S,198594,Qtd Entrada Recoz.,21803.000000
...,...,...,...,...,...,...,...
415,2026-08-10,B,B7,MB -1585-S,198546,%Perdas Recozi.,-0.040799
416,2026-08-10,B,B7,MB -1585-S,198546,%Entrada AF,0.872265
417,2026-08-10,B,B7,MB -1585-S,198546,%Rejeição EA,-0.037930
418,2026-08-10,B,B7,MB -1585-S,198546,%Rejeito AF,-0.138531


In [32]:
df_desemp_emp_diario

,OP Vertech,Data Wht (dia),Maquina,Prefixo,Objetivo %,Empacotado %,Rejeição %,Emp - Rejeitado %
0,198135,2026-08-10,A3,SB -1036-SWN,0.85,0.874625,0.000000,0.874625
1,198176,2026-08-10,14,LB -0478-N1,0.60,0.619792,0.000000,0.619792
2,198456,2026-08-10,15,LB -0491-N1,0.55,0.773889,0.000000,0.773889
3,198546,2026-08-10,B7,MB -1585-S,0.65,0.732791,0.059621,0.689102
4,198567,2026-08-10,A2,SB -1037-SWN,0.87,0.796296,0.000000,0.796296
5,198592,2026-08-10,12,MB -1512-SN,0.65,0.687099,0.000000,0.687099
6,198594,2026-08-10,10,MB -1398-S,0.60,0.589583,0.250000,0.442188
7,198613,2026-08-10,B5,SB -1471-SN,0.00,0.610530,0.093928,0.553184
8,198618,2026-08-10,A1,SB -1035-SWN,0.87,0.891458,0.000000,0.891458
9,198639,2026-08-10,A5,SB -1290-S,0.84,0.937913,0.000000,0.937913
